## 0️. Importar Librerias

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
# Tratamiento de Datos
import pandas as pd
import numpy as np
from IPython.display import display

# Visualizaciones
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Para que se muestren todas las columnas al inspeccionar los DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)


In [3]:
from framework import sp_carga_exploracion as sc
from framework import sp_limpieza_transformacion as sl
from framework import sp_eda as se
from framework import sp_abtest as sb

## 1️. Cargar DataFrame 

In [4]:
ruta = (r"../data/processed/03_datos_eda.csv")
df_ab = sc.leer_csv(ruta)

CSV CARGADO
Archivo: ../data/processed/03_datos_eda.csv
Filas:   1000
Columnas: 14


### escuela copia del  df_ab 

In [5]:
escuela = df_ab.copy()

## 2. Clasificar columnas control - métricas 

### clasificar_columnas

In [6]:
sb.clasificar_columnas(escuela)

,columna,dtype,n_unicos,sugerencia,motivo
0,tiene_tutor_sn,str,2,col_control,Categórica con 2 valores únicos: rango razonable para agrupar
1,aprobado_sn,str,2,col_control,Categórica con 2 valores únicos: rango razonable para agrupar
2,nivel_dificultad,str,3,col_control,Categórica con 3 valores únicos: rango razonable para agrupar
3,horario_estudio_preferido,str,4,col_control,Categórica con 4 valores únicos: rango razonable para agrupar
4,estilo_aprendizaje,str,5,col_control,Categórica con 5 valores únicos: rango razonable para agrupar
5,rendimiento_academico,str,5,col_control,Categórica con 5 valores únicos: rango razonable para agrupar
6,nota_final,float64,364,metrica,Numérica con 364 valores únicos: variable continua medible
7,horas_sueno,float64,811,metrica,Numérica con 811 valores únicos: variable continua medible
8,tasa_asistencia,float64,893,metrica,Numérica con 893 valores únicos: variable continua medible
9,horas_estudio_semanal,float64,964,metrica,Numérica con 964 valores únicos: variable continua medible


##### `target values`

In [7]:
tv_sn ='aprobado_sn'
tv_ml = 'aprobado_ml'

####  `col_control` ¿Qué variable crea grupos?

In [8]:
col_control = ['tiene_tutor_sn','nivel_dificultad','horario_estudio_preferido','estilo_aprendizaje','rendimiento_academico']

`dos_grupos`

In [9]:
dos_grupos = ['tiene_tutor_sn']

`mas_de_dos_grupos`

In [10]:
mas_de_dos_grupos = ['nivel_dificultad','horario_estudio_preferido','estilo_aprendizaje','rendimiento_academico']

#### `metricas`  ¿Qué variable quiero medir?

In [11]:
metricas = ['nota_final','horas_sueno','tasa_asistencia','horas_estudio_semanal','nota_anterior','edad']

## 3. Exploración de Grupos

In [12]:
col_control

['tiene_tutor_sn',
 'nivel_dificultad',
 'horario_estudio_preferido',
 'estilo_aprendizaje',
 'rendimiento_academico']

`tiene_tutor_sn`

In [13]:
sb.exploracion_grupos(escuela,'tiene_tutor_sn',metricas)

,metrica,si,no,diferencia
0,nota_final,75.29,68.84,-6.45
1,horas_sueno,7.15,6.91,-0.24
2,tasa_asistencia,75.86,72.73,-3.13
3,horas_estudio_semanal,13.58,7.71,-5.87
4,nota_anterior,74.47,66.79,-7.68
5,edad,23.67,23.44,-0.23


Hallazgo SG07. Perfil de estudiantes con tutor

Los estudiantes con tutor presentan valores medios superiores en todas las métricas académicas analizadas. Las mayores diferencias se observan en nota anterior (+7,68 puntos), nota final (+6,45 puntos) y horas de estudio semanal (+5,87 horas).

Conclusión SG07

Los alumnos con tutor no solo obtienen mejores resultados académicos, sino que también dedican más tiempo al estudio y presentan un rendimiento previo superior. Las diferencias observadas en edad y horas de sueño son reducidas, por lo que el principal contraste entre ambos grupos parece concentrarse en variables directamente relacionadas con el desempeño académico.

`nivel_dificultad`

In [14]:
sb.exploracion_grupos(escuela,'nivel_dificultad',metricas)

,metrica,facil,dificil,medio,rango
0,nota_final,72.50,69.48,71.49,3.02
2,tasa_asistencia,73.54,74.39,74.13,0.85
4,nota_anterior,69.82,70.50,69.70,0.80
3,horas_estudio_semanal,10.04,9.68,10.24,0.56
5,edad,23.69,23.49,23.45,0.24
1,horas_sueno,7.03,7.13,6.95,0.18


SG08. Perfil por dificultad académica

Hallazgo SG08

La dificultad de la asignatura muestra diferencias reducidas en la mayoría de las métricas analizadas. La mayor variación se observa en la nota final, donde los estudiantes de asignaturas fáciles alcanzan una media de 72,50 puntos frente a los 69,48 puntos de las asignaturas difíciles, con una diferencia de 3,02 puntos.

Conclusión SG08

Las diferencias observadas entre niveles de dificultad son moderadas y no parecen estar asociadas a cambios relevantes en las horas de estudio, la asistencia, la edad o el rendimiento previo. El principal efecto de la dificultad se refleja en el resultado académico final, aumentando ligeramente la probabilidad de obtener calificaciones más bajas en las asignaturas difíciles.

`horario_estudio_preferido`

In [15]:
sb.exploracion_grupos(escuela,'horario_estudio_preferido',metricas)

,metrica,tarde,manana,desconocido,noche,rango
2,tasa_asistencia,76.25,74.20,73.38,71.83,4.42
4,nota_anterior,71.55,69.93,69.57,68.32,3.23
0,nota_final,71.71,71.60,72.37,70.81,1.56
5,edad,23.48,23.27,24.30,23.53,1.03
3,horas_estudio_semanal,10.02,10.57,9.96,9.84,0.73
1,horas_sueno,6.98,7.10,6.84,7.02,0.26


SG09. Perfil según horario de estudio

Hallazgo SG09

Las diferencias observadas entre los distintos horarios de estudio son reducidas en todas las métricas analizadas. La nota final presenta un rango máximo de apenas 1,56 puntos entre categorías, mientras que las horas de estudio semanal muestran diferencias inferiores a una hora.

Conclusión SG09

El horario de estudio preferido no parece constituir un factor diferenciador relevante del rendimiento académico. Los estudiantes presentan perfiles muy similares en términos de nota final, horas de estudio, nota anterior, edad y horas de sueño, independientemente del horario declarado de estudio.

`estilo_aprendizaje`

In [16]:
sb.exploracion_grupos(escuela,'estilo_aprendizaje',metricas)

,metrica,lectura_escritura,desconocido,visual,auditivo,kinestesico,rango
2,tasa_asistencia,77.77,69.55,72.98,74.97,72.62,8.22
4,nota_anterior,71.18,65.05,69.88,70.54,69.19,6.13
0,nota_final,72.59,68.71,70.53,71.82,72.53,3.88
3,horas_estudio_semanal,10.50,9.09,9.69,10.21,10.57,1.48
5,edad,23.50,23.54,23.90,22.96,23.62,0.94
1,horas_sueno,6.94,6.61,7.02,7.05,7.09,0.48


SG10. Perfil según estilo de aprendizaje

Hallazgo SG10

Las diferencias observadas entre estilos de aprendizaje son reducidas en la mayoría de métricas académicas. La mayor variación se encuentra en la tasa de asistencia (8,22 puntos porcentuales) y en la nota anterior (6,13 puntos), mientras que la nota final presenta una diferencia máxima de solo 3,88 puntos entre categorías.

Conclusión SG10

Aunque algunos estilos muestran valores medios ligeramente superiores a otros, no se observa un patrón consistente que permita afirmar que un estilo de aprendizaje concreto se asocie con mejores resultados académicos. Esta conclusión resulta coherente con el análisis previo de significación estadística, donde el estilo de aprendizaje no mostró evidencia suficiente de relación con la variable objetivo.

`rendimiento_academico`

In [17]:
sb.exploracion_grupos(escuela,'rendimiento_academico',metricas)

,metrica,notable (80-89),bien (70-79),suficiente (60-69),suspenso (0-59),sobresaliente (90-100),rango
0,nota_final,83.90,74.69,65.29,55.02,93.46,38.44
4,nota_anterior,77.18,73.15,65.14,58.10,87.37,29.27
2,tasa_asistencia,83.40,75.59,70.54,62.97,81.53,20.43
3,horas_estudio_semanal,13.03,11.07,8.35,5.86,16.84,10.98
5,edad,23.45,23.59,23.45,23.87,22.92,0.95
1,horas_sueno,7.16,6.99,7.02,6.83,6.83,0.33


Hallazgo SG11

Se observa una progresión positiva y continua entre el rendimiento académico y las principales variables académicas del conjunto de datos. A medida que aumenta el nivel de rendimiento, también aumentan la nota anterior, la tasa de asistencia, las horas de estudio semanal y la nota final.

Los estudiantes clasificados como sobresalientes presentan:

Mayor nota anterior (87,37).
Mayor dedicación al estudio (16,84 horas semanales).
Altos niveles de asistencia (81,53%).

Mientras que los alumnos suspensos registran los valores más bajos en todas estas métricas.

Conclusión SG11

El rendimiento académico parece estar principalmente asociado a tres factores:

Mayor nota anterior
+
Mayor dedicación al estudio
+
Mayor asistencia
``

## 4. Flujo de comparación de Grupos

### Normalidad

In [18]:
sb.normalidad(escuela,metricas)

Para la columna NOTA_FINAL los datos SÍ siguen una distribución normal
Para la columna HORAS_SUENO los datos NO siguen una distribución normal
Para la columna TASA_ASISTENCIA los datos NO siguen una distribución normal
Para la columna HORAS_ESTUDIO_SEMANAL los datos NO siguen una distribución normal
Para la columna NOTA_ANTERIOR los datos NO siguen una distribución normal
Para la columna EDAD los datos NO siguen una distribución normal


### homocedasticidad

In [19]:
for col in col_control:
    print("=" * 70)
    print(col.upper())
    print("=" * 70)

    sb.homocedasticidad_resumen(escuela, col, metricas) 

TIENE_TUTOR_SN
Para la columna TIENE_TUTOR_SN las varianzas de las siguientes métricas SÍ son homogéneas. 
SÍ hay HOMOCEDASTICIDAD

['nota_final', 'horas_sueno', 'tasa_asistencia', 'horas_estudio_semanal', 'nota_anterior', 'edad']


NIVEL_DIFICULTAD
Para la columna NIVEL_DIFICULTAD las varianzas de las siguientes métricas SÍ son homogéneas. 
SÍ hay HOMOCEDASTICIDAD

['nota_final', 'horas_sueno', 'tasa_asistencia', 'horas_estudio_semanal', 'nota_anterior', 'edad']


HORARIO_ESTUDIO_PREFERIDO
Para la columna HORARIO_ESTUDIO_PREFERIDO las varianzas de las siguientes métricas SÍ son homogéneas. 
SÍ hay HOMOCEDASTICIDAD

['nota_final', 'horas_sueno', 'tasa_asistencia', 'horas_estudio_semanal', 'nota_anterior', 'edad']


ESTILO_APRENDIZAJE
Para la columna ESTILO_APRENDIZAJE las varianzas de las siguientes métricas SÍ son homogéneas. 
SÍ hay HOMOCEDASTICIDAD

['nota_final', 'horas_sueno', 'tasa_asistencia', 'nota_anterior', 'edad']


RENDIMIENTO_ACADEMICO
Para la columna RENDIMIENTO_ACADEMICO

H700. Validación de homocedasticidad

Las pruebas de homogeneidad de varianzas muestran que la mayoría de grupos analizados presentan varianzas equivalentes para las métricas estudiadas. Este resultado permite aplicar pruebas comparativas con mayor robustez en la fase de AB Testing, especialmente para las variables relacionadas con tutor, dificultad, horario de estudio y estilo de aprendizaje.

## Grupos en col_control

In [20]:
col_control

['tiene_tutor_sn',
 'nivel_dificultad',
 'horario_estudio_preferido',
 'estilo_aprendizaje',
 'rendimiento_academico']

In [21]:
dos_grupos

['tiene_tutor_sn']

In [22]:
mas_de_dos_grupos

['nivel_dificultad',
 'horario_estudio_preferido',
 'estilo_aprendizaje',
 'rendimiento_academico']

In [23]:
metricas

['nota_final',
 'horas_sueno',
 'tasa_asistencia',
 'horas_estudio_semanal',
 'nota_anterior',
 'edad']

 ### ttests_dos_grupos

In [24]:
sb.ttest_dos_grupos(escuela,'tiene_tutor_sn',['nota_final'])

t=11.1136, p=0.0000
Para la métrica NOTA_FINAL, las medias NO son iguales, es decir, SI hay diferencias significativas entre grupos


In [25]:
sb.mannwhitneyu(escuela,'tiene_tutor_sn','nota_final')

U=166747.5000, p=0.0000
Para la métrica NOTA_FINAL, las medianas NO son iguales, es decir, SI hay deferencias significativas entre grupos


**Hipótesis**<br>
H0: La nota final media es igual en ambos grupos.<br>
H1: La nota final media es diferente en ambos grupos.<br>

**Resultados**<br>
t-test:<br>
t = 11.1136<br>
p < 0.001

Mann-Whitney:<br>
U = 166747.5<br>
p < 0.001

Se rechaza la hipótesis nula. Existen diferencias
estadísticamente significativas en la nota final
entre los estudiantes con tutor y los estudiantes
sin tutor.

Los alumnos con tutor presentan una nota final
media superior (75,29 puntos) a la observada en
los alumnos sin tutor (68,84 puntos).

### mannwhitneyu

In [26]:
sb.mannwhitneyu(escuela,'tiene_tutor_sn',['horas_sueno','tasa_asistencia','horas_estudio_semanal','nota_anterior','edad'])

U=133575.5000, p=0.0030
Para la métrica HORAS_SUENO, las medianas NO son iguales, es decir, SI hay deferencias significativas entre grupos
U=132330.0000, p=0.0072
Para la métrica TASA_ASISTENCIA, las medianas NO son iguales, es decir, SI hay deferencias significativas entre grupos
U=205110.5000, p=0.0000
Para la métrica HORAS_ESTUDIO_SEMANAL, las medianas NO son iguales, es decir, SI hay deferencias significativas entre grupos
U=155376.0000, p=0.0000
Para la métrica NOTA_ANTERIOR, las medianas NO son iguales, es decir, SI hay deferencias significativas entre grupos
U=124677.0000, p=0.3264
Para la métrica EDAD, las medianas SI son iguales, es decir, NO hay deferencias significativas entre grupos


AB01. Impacto de la disponibilidad de tutor

Se han encontrado diferencias estadísticamente significativas entre estudiantes con y sin tutor en las siguientes métricas:

Nota final (p < 0,001)<br>
Horas de sueño (p = 0,003)<br>
Tasa de asistencia (p = 0,007)<br>
Horas de estudio semanal (p < 0,001)<br>
Nota anterior (p < 0,001)<br>

Por el contrario, la edad no presenta diferencias estadísticamente significativas entre ambos grupos (p = 0,326).

Conclusión

Los estudiantes con tutor no sólo obtienen mejores resultados académicos, sino que también presentan un perfil distinto caracterizado por:<br>
Más horas de estudiobr<br>
Mayor nota previabr<br>
Mayor nota finalbr<br>
Mayor asistenciabr<br>
Ligeramente más horas de sueño<br>

La edad no parece diferenciar a ambos grupos.

### kruskal

In [27]:
kruskal_res = sb.kruskal(
    escuela,
    mas_de_dos_grupos,
    metricas
)

kruskal_res

,control,metrica,H,pvalue,significativo
0,nivel_dificultad,nota_final,12.3261,0.0021,True
1,nivel_dificultad,horas_sueno,2.6278,0.2688,False
2,nivel_dificultad,tasa_asistencia,0.3454,0.8414,False
3,nivel_dificultad,horas_estudio_semanal,2.4172,0.2986,False
4,nivel_dificultad,nota_anterior,0.1045,0.9491,False
5,nivel_dificultad,edad,0.8774,0.6449,False
6,horario_estudio_preferido,nota_final,2.1549,0.5409,False
7,horario_estudio_preferido,horas_sueno,4.1524,0.2455,False
8,horario_estudio_preferido,tasa_asistencia,10.1888,0.0170,True
9,horario_estudio_preferido,horas_estudio_semanal,2.9358,0.4016,False


In [28]:
kruskal_res.query(
    'significativo == True'
).sort_values(
    'pvalue'
)

,control,metrica,H,pvalue,significativo
18,rendimiento_academico,nota_final,904.3676,0.0000,True
20,rendimiento_academico,tasa_asistencia,99.2079,0.0000,True
21,rendimiento_academico,horas_estudio_semanal,237.1211,0.0000,True
22,rendimiento_academico,nota_anterior,182.7878,0.0000,True
0,nivel_dificultad,nota_final,12.3261,0.0021,True
8,horario_estudio_preferido,tasa_asistencia,10.1888,0.0170,True
12,estilo_aprendizaje,nota_final,11.9887,0.0174,True
14,estilo_aprendizaje,tasa_asistencia,11.3586,0.0228,True
17,estilo_aprendizaje,edad,11.1176,0.0253,True
10,horario_estudio_preferido,nota_anterior,8.1160,0.0437,True


Las métricas significativas son las métricas que cambian cuando cambian los grupos.

In [29]:
(
    kruskal_res.query('significativo')
       .groupby('control')
       .size()
       .sort_values(ascending=False)
)

control
rendimiento_academico        4
estilo_aprendizaje           3
horario_estudio_preferido    2
nivel_dificultad             1
dtype: int64

In [30]:
(
    kruskal_res.query('significativo')
       .groupby('metrica')
       .size()
       .sort_values(ascending=False)
)

metrica
nota_final               3
tasa_asistencia          3
nota_anterior            2
edad                     1
horas_estudio_semanal    1
dtype: int64

### decidir_test

In [35]:
res = sb.decidir_test(escuela,col_control,lista_metricas=metricas)

In [37]:
res

,control,metrica,test,estadistico,pvalue,normal,homocedastico,n_grupos,significativo
0,tiene_tutor_sn,nota_final,ttest,-11.1136,0.0000,True,True,2,True
1,tiene_tutor_sn,horas_sueno,mannwhitney,107015.5000,0.0030,False,True,2,True
2,tiene_tutor_sn,tasa_asistencia,mannwhitney,108261.0000,0.0072,False,True,2,True
3,tiene_tutor_sn,horas_estudio_semanal,mannwhitney,35480.5000,0.0000,False,True,2,True
4,tiene_tutor_sn,nota_anterior,mannwhitney,85215.0000,0.0000,False,True,2,True
5,tiene_tutor_sn,edad,mannwhitney,115914.0000,0.3264,False,True,2,False
6,nivel_dificultad,nota_final,anova,5.8578,0.0030,True,True,3,True
7,nivel_dificultad,horas_sueno,kruskal,2.6278,0.2688,False,True,3,False
8,nivel_dificultad,tasa_asistencia,kruskal,0.3454,0.8414,False,True,3,False
9,nivel_dificultad,horas_estudio_semanal,kruskal,2.4172,0.2986,False,True,3,False


In [36]:
res.query(
    'significativo == True'
).sort_values(
    'pvalue'
)

,control,metrica,test,estadistico,pvalue,normal,homocedastico,n_grupos,significativo
0,tiene_tutor_sn,nota_final,ttest,-11.1136,0.0000,True,True,2,True
3,tiene_tutor_sn,horas_estudio_semanal,mannwhitney,35480.5000,0.0000,False,True,2,True
4,tiene_tutor_sn,nota_anterior,mannwhitney,85215.0000,0.0000,False,True,2,True
24,rendimiento_academico,nota_final,kruskal,904.3676,0.0000,True,False,5,True
26,rendimiento_academico,tasa_asistencia,kruskal,99.2079,0.0000,False,False,5,True
27,rendimiento_academico,horas_estudio_semanal,kruskal,237.1211,0.0000,False,True,5,True
28,rendimiento_academico,nota_anterior,kruskal,182.7878,0.0000,False,True,5,True
1,tiene_tutor_sn,horas_sueno,mannwhitney,107015.5000,0.0030,False,True,2,True
6,nivel_dificultad,nota_final,anova,5.8578,0.0030,True,True,3,True
2,tiene_tutor_sn,tasa_asistencia,mannwhitney,108261.0000,0.0072,False,True,2,True


# Conclusiones finales del AB Testing

El análisis estadístico automatizado confirma los principales hallazgos identificados previamente durante el EDA.

## Variables con mayor evidencia estadística

### Tiene tutor (`tiene_tutor_sn`)

Se observan diferencias estadísticamente significativas en:

- Nota final (p < 0.001)
- Horas de estudio semanal (p < 0.001)
- Nota anterior (p < 0.001)
- Horas de sueño (p = 0.003)
- Tasa de asistencia (p = 0.007)

No se observan diferencias significativas en:

- Edad (p = 0.326)

Los estudiantes con tutor presentan un perfil académico más favorable, caracterizado por mayor dedicación al estudio, mejores resultados previos y mejores calificaciones finales.

---

### Nivel de dificultad (`nivel_dificultad`)

Únicamente se detectan diferencias significativas en:

- Nota final (p = 0.003)

No se observan diferencias significativas en el resto de métricas analizadas.

La dificultad parece afectar principalmente al resultado académico final.

---

### Horario de estudio preferido (`horario_estudio_preferido`)

Se observan diferencias significativas en:

- Tasa de asistencia (p = 0.017)
- Nota anterior (p = 0.044)

No se detectan diferencias significativas en:

- Nota final
- Horas de sueño
- Horas de estudio semanal
- Edad

Su impacto sobre el rendimiento académico parece limitado.

---

